In [1]:
import torch,re
import time
from tqdm import tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import librosa 


In [2]:
# Kaynaklarda kullanılan temel ayarlar [4]
model_id = "openai/whisper-large-v3"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32


In [3]:
# 1. Model ve Processor'ı Doğrudan Yükleme [1]
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
).to(device)

processor = AutoProcessor.from_pretrained(model_id)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [4]:

# Not: Ses dosyasını yüklemek için kaynaklarda belirtilmeyen 'librosa' gibi 
# harici bir kütüphane kullanmanız gerekebilir (Bu bilgi kaynak dışıdır).

audio_path = "seeyou.mp3"
speech, sr = librosa.load(audio_path, sr=16000) # Whisper 16kHz ile çalışır

In [5]:
generate_kwargs = {
    "max_length": 448,  # Uyarıyı kapatan kritik satır
    "num_beams": 1,
    "condition_on_prev_tokens": False, # Önceki tokenlara bağımlılığı kapatır [1]
    "compression_ratio_threshold": 1.35,
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0), # Temperature fallback stratejisi [1], [3]
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.5,
    "return_timestamps": True, # Zaman damgalarını tahmin etmesini sağlar [2]
    "language":"english"
}

In [6]:
# 2. Parçalama Mantığı (Chunking) [2, 3]
# Whisper 30 saniyelik pencerelerle çalıştığı için veriyi buna göre böleceğiz.
chunk_length_s = 30
samples_per_chunk = chunk_length_s * sr
total_chunks = (len(speech) // samples_per_chunk) + 1
print(f"Toplam {total_chunks} parça işlenecek...")


def text_Totimestamps(text,start_time=0) :
        print("start_time : ",start_time)
        pattern = r"<\|(\d+\.\d+)\|>(.*?)<\|(\d+\.\d+)\|>"
        matches = re.findall(pattern, text)
        # İstenen format: [[[zaman damgası], [text]]]
        formatted_output = [{"timestamp":(float(m[0])+start_time, float(m[2])+start_time),
                             "text":(m[1].strip())} for m in matches]

        return formatted_output


Toplam 8 parça işlenecek...


In [7]:
# 3. Manuel İlerleme Çubuğu ile Döngü
with tqdm(total=total_chunks) as pbar:
    final_text=[]
    full_transcript=[]
    for s,i in enumerate(range(0, len(speech), samples_per_chunk)):
        # Ses parçasını al
        chunk = speech[i : i + samples_per_chunk]
        
        # Processor: Ses verisini modelin anlayacağı Mel-spektrograma çevirir [1, 5]
        input_features = processor(chunk, sampling_rate=sr, return_tensors="pt").input_features
        input_features = input_features.to(device).to(torch_dtype)
        
        # Model.generate: Metin üretim aşaması [1]
        with torch.no_grad():
            predicted_ids = model.generate(input_features,**generate_kwargs) # Dil seçimi [6]
            
        # Decode: Sayısal çıktıları metne dönüştürür
        transcription = processor.batch_decode(*predicted_ids, skip_special_tokens=True,decode_with_timestamps=True)
        
        formatted_output=text_Totimestamps(transcription[0],start_time=s*30)

        torch.cuda.empty_cache()
        # Her parça bittiğinde ilerleme çubuğunu güncelle

        final_text+=formatted_output
        pbar.update(1)

        
        

print("\nİşlem Tamamlandı.")

  0%|                             | 0/8 [00:00<?, ?it/s][transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, 

start_time :  0
start_time (s) :  0
transcription :  ["<|0.00|> It's been a long day without you, my friend<|16.78|><|16.78|> And I'll tell you all about it when I see you again<|22.74|><|22.74|> We've come a long way from where we began<|28.82|><|28.82|> Oh, I'll<|29.98|>"]


 25%|█████▎               | 2/8 [00:05<00:15,  2.64s/it]

start_time :  30
start_time (s) :  1
transcription :  ["<|0.00|> Tell you all about it when I see you again<|4.74|><|4.74|> When I see you again<|7.70|><|7.70|> Damn, who knew?<|11.48|><|11.74|> All the planes we flew<|12.98|><|12.98|> Good things we've been through<|14.74|><|14.74|> That I'd be standing right here talking to you<|17.48|><|17.48|> About another path<|18.86|><|18.86|> I know we love to hit the road and laugh<|21.08|><|21.08|> But something told me that it wouldn't last<|23.46|><|23.46|> Had to switch up, look at things different<|25.88|><|25.88|> See the bigger picture<|27.08|><|27.08|> Those were the days, hard work forever paid<|29.98|>"]


 38%|███████▉             | 3/8 [00:07<00:13,  2.72s/it]

...

start_time :  150
start_time (s) :  5
transcription :  ["<|0.00|> Let the light guide your way<|4.00|><|6.00|> Hold every memory as you go<|10.00|><|11.00|> And every road you take<|14.00|><|15.00|> Will always lead you home<|19.00|><|23.00|> It's been a long day<|26.00|><|27.00|> Without you my friend<|29.00|>"]


 88%|██████████████████▍  | 7/8 [00:18<00:02,  2.84s/it]

start_time :  180
start_time (s) :  6
transcription :  ["<|0.00|> I'll tell you all about it when I see you again<|5.00|><|5.00|> We've come a long way from where we began<|11.00|><|11.00|> Oh, I'll tell you all about it when I see you again<|17.00|><|17.00|> When I see you again<|21.00|><|21.00|> Oh, oh, oh<|30.00|>"]


100%|█████████████████████| 8/8 [00:19<00:00,  2.40s/it]

start_time :  210
start_time (s) :  7
transcription :  ['<|0.00|> See you again, see you again<|4.16|><|4.16|> See you again<|13.60|>']

İşlem Tamamlandı.


In [2]:
import os

os.chdir("..")
print(os.getcwd())

E:\PROJECT\Yapay_zeka_sesten_yazıya


In [3]:
from model import WhisperLargeV3
from tools import data_to_srt

In [4]:
WhisperModel=WhisperLargeV3()

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [3]:
#WhisperModel.Load_model()

In [5]:
#WhisperModel.language="turkish"

In [6]:
speech, sr = WhisperModel.Load_file("data/kesit.mp3")

In [7]:
result=WhisperModel.preadiction(speech=speech, sr=sr)

  0%|                                                                | 0/2 [00:00<?, ?it/s][transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it 

In [8]:
data_to_srt(result,output_file="mahsun.srt")

In [9]:
# Bu kod kaynaklarda bulunmamaktadır.
from moviepy import VideoFileClip



In [10]:
video = VideoFileClip("mahsun.ts")
video.audio.write_audiofile("mahsun.mp3") # Sesi mp3 olarak kaydeder

MoviePy - Writing audio in mahsun.mp3


MoviePy - Done.
